In [41]:
from pathlib import Path
from pyspark.sql import SparkSession
import pandas as pd
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"

data_dict = pd.read_csv(str(DATA_DIR / "diccionario_datos.csv"), 
                        encoding="latin-1",
                        sep=";")
spark = SparkSession.builder.appName("eda").getOrCreate()
crimes = spark.read.csv(str(DATA_DIR / "delitos_2015_2025.csv"), 
                        header=True, inferSchema=True, encoding="iso-8859-1")




In [42]:
#ver descripciones de las columnas
print(type(data_dict))
#la clave de la entidad la podemos relacionar con endireh y con ingei
data_dict["Unnamed: 1"][2]

<class 'pandas.DataFrame'>


'Clave de la entidad, según el Marco Geoestadístico Nacional (MGN) del Instituto Nacional de Geografía y Estadística (INEGI).'

In [43]:
mapeo = {
    "Año": "anio",
    "Clave_Ent": "clave_ent",
    "Entidad": "entidad",
    "Bien jurídico afectado": "bien_juridico",
    "Tipo de delito": "tipo_delito",
    "Subtipo de delito": "subtipo_delito",
    "Modalidad": "modalidad",
    "Enero": "enero",
    "Febrero": "febrero",
    "Marzo": "marzo",
    "Abril": "abril",
    "Mayo": "mayo",
    "Junio": "junio",
    "Julio": "julio",
    "Agosto": "agosto",
    "Septiembre": "septiembre",
    "Octubre": "octubre",
    "Noviembre": "noviembre",
    "Diciembre": "diciembre",
}

crimes = crimes.withColumnsRenamed(mapeo)
crimes.printSchema()

root
 |-- anio: integer (nullable = true)
 |-- clave_ent: integer (nullable = true)
 |-- entidad: string (nullable = true)
 |-- bien_juridico: string (nullable = true)
 |-- tipo_delito: string (nullable = true)
 |-- subtipo_delito: string (nullable = true)
 |-- modalidad: string (nullable = true)
 |-- enero: integer (nullable = true)
 |-- febrero: integer (nullable = true)
 |-- marzo: integer (nullable = true)
 |-- abril: integer (nullable = true)
 |-- mayo: integer (nullable = true)
 |-- junio: integer (nullable = true)
 |-- julio: integer (nullable = true)
 |-- agosto: integer (nullable = true)
 |-- septiembre: integer (nullable = true)
 |-- octubre: integer (nullable = true)
 |-- noviembre: integer (nullable = true)
 |-- diciembre: integer (nullable = true)



In [44]:
from pyspark.sql import functions as F
#pero es mejor ponerlo en formato largo
months = crimes.columns[7:]
lcrimes = (
    crimes.unpivot(
        ids=["anio", "clave_ent", "entidad", "bien_juridico",
            "tipo_delito", "subtipo_delito", "modalidad"],
        values=months, 
        variableColumnName="mes",
        valueColumnName="incidencia"
    )
    .withColumn("num_mes", F.array_position(F.array(*[F.lit(m) for m in months]), F.col("mes")))
    .withColumn("fecha", F.make_date("anio", "num_mes", F.lit(1)))
)



                    

In [45]:
lcrimes.groupBy("tipo_delito").count().show(100, truncate=False)
#candidatos: secuestro
studied_crimes = ["Abuso sexual", "Violencia familiar", 
                  "Otros delitos que atentan contra la vida y la integridad corporal",
                  "Violencia de género en todas sus modalidades distinta a la violencia familiar",
                  "Acoso sexual", "Violación equiparada", 
                  "Otros delitos que atentan contra la libertad y la seguridad sexual",
                  "Feminicidio", "Violación simple", "Hostigamiento sexual", 
                  "Trata de personas"]

#NOTA: pero no sabemos si todos peretencen a mujeres
fem_crimes = lcrimes.where(F.col("tipo_delito").isin(studied_crimes))

+-----------------------------------------------------------------------------+------+
|tipo_delito                                                                  |count |
+-----------------------------------------------------------------------------+------+
|Abuso sexual                                                                 |4224  |
|Fraude                                                                       |4224  |
|Tráfico de menores                                                           |4224  |
|Allanamiento de morada                                                       |4224  |
|Violencia familiar                                                           |4224  |
|Otros delitos que atentan contra la vida y la integridad corporal            |4224  |
|Violencia de género en todas sus modalidades distinta a la violencia familiar|4224  |
|Otros delitos contra la sociedad                                             |4224  |
|Delitos cometidos por servidores públicos 

Nota: no se pudo separar de los delitos por sexo 

In [46]:
fem_crimes.show()

+----+---------+--------------+--------------------+-----------+--------------+-----------------+----------+----------+-------+----------+
|anio|clave_ent|       entidad|       bien_juridico|tipo_delito|subtipo_delito|        modalidad|       mes|incidencia|num_mes|     fecha|
+----+---------+--------------+--------------------+-----------+--------------+-----------------+----------+----------+-------+----------+
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|     enero|         0|      1|2015-01-01|
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|   febrero|         0|      2|2015-02-01|
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|     marzo|         0|      3|2015-03-01|
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|     abril|         0|      4|2015-04-01|
|2015|        1|Aguascalien

In [47]:
cases = fem_crimes.groupBy("incidencia").count().where(F.col("incidencia") > 0).agg(F.sum("count").alias("total casos"))
cero_cases = fem_crimes.groupBy("incidencia").count().where(F.col("incidencia") == 0).agg(F.sum("count").alias("total ceros"))
total = cases.first()["total casos"] + cero_cases.first()["total ceros"]
print(f"columnas totales = {total}")
print(fem_crimes.count() == total)
cases.show()
cero_cases.show()

columnas totales = 59136
True
+-----------+
|total casos|
+-----------+
|      35571|
+-----------+

+-----------+
|total ceros|
+-----------+
|      23565|
+-----------+



Ahora vamos con la poblacionp para poder calcular un indice de gini

In [48]:
pred_population = spark.read.csv(str(DATA_DIR / "prediccion_poblacion.csv"), header=True)
pred_population.show()

+-------+----+--------------+-------+----+-------+---------+------------------+----------+
|RENGLON|ANIO|       ENTIDAD|CVE_GEO|EDAD|   SEXO|POBLACION|ENTIDAD_FEDERATIVA|     FECHA|
+-------+----+--------------+-------+----+-------+---------+------------------+----------+
|   4621|1970|Aguascalientes|      1|   0|Hombres|     7450|    Aguascalientes|1970-01-01|
|   4622|1970|Aguascalientes|      1|   0|Mujeres|     7251|    Aguascalientes|1970-01-01|
|   4623|1970|Aguascalientes|      1|   1|Hombres|     7072|    Aguascalientes|1970-01-01|
|   4624|1970|Aguascalientes|      1|   1|Mujeres|     6902|    Aguascalientes|1970-01-01|
|   4625|1970|Aguascalientes|      1|   2|Hombres|     6849|    Aguascalientes|1970-01-01|
|   4626|1970|Aguascalientes|      1|   2|Mujeres|     6680|    Aguascalientes|1970-01-01|
|   4627|1970|Aguascalientes|      1|   3|Hombres|     6660|    Aguascalientes|1970-01-01|
|   4628|1970|Aguascalientes|      1|   3|Mujeres|     6493|    Aguascalientes|1970-01-01|

In [49]:
population = pred_population.toDF(*[c.lower() for c in pred_population.columns])
population = (
    population
        .where(
            F.col("anio").between(2015, 2025) & 
            (F.col("sexo") == "Mujeres")
        )
        .groupBy("entidad", "cve_geo", "fecha")
        .agg(F.sum("poblacion").alias("poblacion"))
        .orderBy(F.asc("entidad"), F.asc("fecha"))
)
#ajuste para el join en la construccion de tasas
population = population.withColumnRenamed("cve_geo", "clave_ent")
population = (
    population
        .withColumn("anio", F.year("fecha"))
        .drop("fecha")
)
population.show()

+---------------+---------+---------+----+
|        entidad|clave_ent|poblacion|anio|
+---------------+---------+---------+----+
| Aguascalientes|        1| 684205.0|2015|
| Aguascalientes|        1| 695776.0|2016|
| Aguascalientes|        1| 707389.0|2017|
| Aguascalientes|        1| 719280.0|2018|
| Aguascalientes|        1| 731399.0|2019|
| Aguascalientes|        1| 741928.0|2020|
| Aguascalientes|        1| 750895.0|2021|
| Aguascalientes|        1| 760442.0|2022|
| Aguascalientes|        1| 770393.0|2023|
| Aguascalientes|        1| 780169.0|2024|
| Aguascalientes|        1| 789763.0|2025|
|Baja California|        2|1696306.0|2015|
|Baja California|        2|1735383.0|2016|
|Baja California|        2|1777162.0|2017|
|Baja California|        2|1819712.0|2018|
|Baja California|        2|1862700.0|2019|
|Baja California|        2|1896030.0|2020|
|Baja California|        2|1920718.0|2021|
|Baja California|        2|1948170.0|2022|
|Baja California|        2|1977058.0|2023|
+----------

In [50]:
fem_crimes.show()

+----+---------+--------------+--------------------+-----------+--------------+-----------------+----------+----------+-------+----------+
|anio|clave_ent|       entidad|       bien_juridico|tipo_delito|subtipo_delito|        modalidad|       mes|incidencia|num_mes|     fecha|
+----+---------+--------------+--------------------+-----------+--------------+-----------------+----------+----------+-------+----------+
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|     enero|         0|      1|2015-01-01|
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|   febrero|         0|      2|2015-02-01|
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|     marzo|         0|      3|2015-03-01|
|2015|        1|Aguascalientes|La vida y la Inte...|Feminicidio|   Feminicidio|Con arma de fuego|     abril|         0|      4|2015-04-01|
|2015|        1|Aguascalien

In [51]:

#delitos generales por cada estado
crimes_per_state = (
    fem_crimes
        .groupBy(
            "clave_ent", "entidad", F.year("fecha").alias("anio")
        )
        .agg(F.sum("incidencia").alias("delitos"))
        .orderBy(
            F.asc("entidad"), F.asc("anio")
        )
)
crimes_per_state.show(100)

+---------+--------------------+----+-------+
|clave_ent|             entidad|anio|delitos|
+---------+--------------------+----+-------+
|        1|      Aguascalientes|2015|    961|
|        1|      Aguascalientes|2016|   1046|
|        1|      Aguascalientes|2017|   1667|
|        1|      Aguascalientes|2018|   2285|
|        1|      Aguascalientes|2019|   2981|
|        1|      Aguascalientes|2020|   2986|
|        1|      Aguascalientes|2021|   3346|
|        1|      Aguascalientes|2022|   3687|
|        1|      Aguascalientes|2023|   4543|
|        1|      Aguascalientes|2024|   4587|
|        1|      Aguascalientes|2025|   3922|
|        2|     Baja California|2015|  11868|
|        2|     Baja California|2016|  11535|
|        2|     Baja California|2017|  11923|
|        2|     Baja California|2018|  13609|
|        2|     Baja California|2019|  14556|
|        2|     Baja California|2020|  15439|
|        2|     Baja California|2021|  17878|
|        2|     Baja California|20

Medicion general del nivel de violencia por cada estado, sin segmentar por: 
- grupos de edad
- tipo de delito


In [52]:
#ahora si la tasa por cada estaod y anio
tasas = (
    crimes_per_state
        .join(population.select("clave_ent", "anio", "poblacion"), on=["clave_ent", "anio"], how = "left")
        .withColumn("tasa_delitos_100k", F.round((F.col("delitos") / F.col("poblacion")) * 100_000, 2))
        .orderBy(F.asc("entidad"), F.asc("anio"))
)
tasas.show()
    


+---------+----+---------------+-------+---------+-----------------+
|clave_ent|anio|        entidad|delitos|poblacion|tasa_delitos_100k|
+---------+----+---------------+-------+---------+-----------------+
|        1|2015| Aguascalientes|    961| 684205.0|           140.45|
|        1|2016| Aguascalientes|   1046| 695776.0|           150.34|
|        1|2017| Aguascalientes|   1667| 707389.0|           235.66|
|        1|2018| Aguascalientes|   2285| 719280.0|           317.68|
|        1|2019| Aguascalientes|   2981| 731399.0|           407.58|
|        1|2020| Aguascalientes|   2986| 741928.0|           402.46|
|        1|2021| Aguascalientes|   3346| 750895.0|            445.6|
|        1|2022| Aguascalientes|   3687| 760442.0|           484.85|
|        1|2023| Aguascalientes|   4543| 770393.0|            589.7|
|        1|2024| Aguascalientes|   4587| 780169.0|           587.95|
|        1|2025| Aguascalientes|   3922| 789763.0|            496.6|
|        2|2015|Baja California|  

Aqui analizaremos los delitos solo en mujeres


In [55]:
victims_data = spark.read.csv(str(DATA_DIR / "victimas_2015_2025.csv"), 
                        header=True, inferSchema=True, encoding="iso-8859-1")
victims_data.show()

+----+---------+--------------+----------------------+--------------+-----------------+--------------------+-----+--------------------+-----+-------+-----+-----+----+-----+-----+------+----------+-------+---------+---------+
| Año|Clave_Ent|       Entidad|Bien jurídico afectado|Tipo de delito|Subtipo de delito|           Modalidad| Sexo|       Rango de edad|Enero|Febrero|Marzo|Abril|Mayo|Junio|Julio|Agosto|Septiembre|Octubre|Noviembre|Diciembre|
+----+---------+--------------+----------------------+--------------+-----------------+--------------------+-----+--------------------+-----+-------+-----+-----+----+-----+-----+------+----------+-------+---------+---------+
|2015|        1|Aguascalientes|  La vida y la Inte...|     Homicidio| Homicidio doloso|   Con arma de fuego|Mujer|Menores de edad (...|    0|      0|    0|    0|   0|    0|    0|     0|         0|      0|        0|        0|
|2015|        1|Aguascalientes|  La vida y la Inte...|     Homicidio| Homicidio doloso|     Con arma

In [58]:

mapping = {
    "Año": "anio",
    "Clave_Ent": "clave_ent",
    "Entidad": "entidad",
    "Bien jurídico afectado": "bien_juridico",
    "Tipo de delito": "tipo_delito",
    "Subtipo de delito": "subtipo_delito",
    "Modalidad": "modalidad"
}
victims = victims_data.withColumnsRenamed(mapping)
victims = victims.toDF(*[c.lower().replace(" ", "_") for c in victims.columns])
final_cols = [x for x in victims.columns if x not in months]
victims = (victims.unpivot(
    ids= list(set(victims.columns) - set(months)),
    values=months,
    variableColumnName="mes",
    valueColumnName="incidencia",
    )
    .select(final_cols)
)
victims.show()

+----+---------+--------------+--------------------+-----------+----------------+-----------------+-----+--------------------+
|anio|clave_ent|       entidad|       bien_juridico|tipo_delito|  subtipo_delito|        modalidad| sexo|       rango_de_edad|
+----+---------+--------------+--------------------+-----------+----------------+-----------------+-----+--------------------+
|2015|        1|Aguascalientes|La vida y la Inte...|  Homicidio|Homicidio doloso|Con arma de fuego|Mujer|Menores de edad (...|
|2015|        1|Aguascalientes|La vida y la Inte...|  Homicidio|Homicidio doloso|Con arma de fuego|Mujer|Menores de edad (...|
|2015|        1|Aguascalientes|La vida y la Inte...|  Homicidio|Homicidio doloso|Con arma de fuego|Mujer|Menores de edad (...|
|2015|        1|Aguascalientes|La vida y la Inte...|  Homicidio|Homicidio doloso|Con arma de fuego|Mujer|Menores de edad (...|
|2015|        1|Aguascalientes|La vida y la Inte...|  Homicidio|Homicidio doloso|Con arma de fuego|Mujer|Menore

In [62]:
#existe el sexo: no identificado
fem_victims = victims.where(F.col("sexo") == "Mujer")

#ver cuantos tipos de crimenes se reportaron
fem_victims.groupBy("tipo_delito").count().show(100, truncate=False)

+-----------------------------------------------------------------+------+
|tipo_delito                                                      |count |
+-----------------------------------------------------------------+------+
|Tráfico de menores                                               |12672 |
|Otros delitos que atentan contra la vida y la integridad corporal|12672 |
|Otros delitos contra la sociedad                                 |12672 |
|Secuestro                                                        |63360 |
|Trata de personas                                                |12672 |
|Otros delitos que atentan contra la libertad personal            |12672 |
|Corrupción de menores                                            |12672 |
|Rapto                                                            |12672 |
|Lesiones                                                         |114048|
|Extorsión                                                        |12672 |
|Feminicidio             

In [ ]:
#tipo de delito por cada entidad por cada anio
crimes_per_state_year = (fem_victims
    .groupBy(
        "entidad", 
        "anio", 
        "tipo_delito"
    )
    .count()
    .orderBy(
        F.asc("entidad"), 
        F.asc("anio")
    ).show()
)

+--------------+----+--------------------+-----+
|       entidad|anio|         tipo_delito|count|
+--------------+----+--------------------+-----+
|Aguascalientes|2015|            Lesiones|  324|
|Aguascalientes|2015|  Tráfico de menores|   36|
|Aguascalientes|2015|Otros delitos con...|   36|
|Aguascalientes|2015|   Trata de personas|   36|
|Aguascalientes|2015|           Extorsión|   36|
|Aguascalientes|2015|               Rapto|   36|
|Aguascalientes|2015|Otros delitos que...|   36|
|Aguascalientes|2015|           Secuestro|  180|
|Aguascalientes|2015|         Feminicidio|  144|
|Aguascalientes|2015|           Homicidio|  324|
|Aguascalientes|2015|Corrupción de men...|   36|
|Aguascalientes|2015|Otros delitos que...|   36|
|Aguascalientes|2016|            Lesiones|  324|
|Aguascalientes|2016|           Secuestro|  180|
|Aguascalientes|2016|           Homicidio|  324|
|Aguascalientes|2016|Otros delitos que...|   36|
|Aguascalientes|2016|         Feminicidio|  144|
|Aguascalientes|2016

Nota de posible investigacion: la cantidad de armas que se emplean y como es que se tienen esa 
cantidad de armas